# Model Comparison: CNNLSTM vs TCN
Compare different neural network architectures for exercise classification from IMU data.

## 1. Environment Setup and Reproducibility
Load dependencies, set random seeds, and configure device.

In [ ]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_pipeline import (
    IMU_FEATURES,
    clean_imu_columns,
    encode_activities,
    load_filtered_recordings,
    make_train_test_loaders,
)
from model_architecture import CNNLSTM, TCN
from model_comparison import ModelComparison
from train_eval import get_device

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configuration
CONFIG = {
    "data_path": "data",
    "window_size": 300,
    "step_size": 100,
    "train_split": 0.8,
    "batch_size_train": 32,
    "batch_size_test": 1,
    "num_epochs": 50,
    "learning_rate": 2e-4,
    "clip_grad_norm": 1.0,
    "patience": 7,
}

# Get device
device = get_device()
print(f"Using device: {device}")

## 2. Load and Inspect Dataset
Load IMU recordings and check data structure and class distribution.

In [ ]:
# Load and inspect data
data = load_filtered_recordings(data_path=CONFIG["data_path"], min_recordings_per_activity=5)
activity_to_id = encode_activities(data)
clean_imu_columns(data, IMU_FEATURES)

# Dataset statistics
segment_lengths = [len(df) for df in data]
print(f"Dataset loaded:")
print(f"  Total recordings: {len(data)}")
print(f"  Unique activities: {len(activity_to_id)}")
print(f"  Activities: {list(activity_to_id.keys())}")
print(f"  Segment length - min: {min(segment_lengths)}, median: {int(np.median(segment_lengths))}, max: {max(segment_lengths)}")
print(f"  IMU features ({len(IMU_FEATURES)}): {IMU_FEATURES}")

id_to_activity = {idx: activity for activity, idx in activity_to_id.items()}

## 3. Preprocess Data and Build Time Windows
Create sliding windows from the time-series IMU data for supervised learning.

In [ ]:
# Build train/test loaders with windowed data
train_loader, test_loader, train_dataset, test_dataset = make_train_test_loaders(
    data=data,
    imu_features=IMU_FEATURES,
    window_size=CONFIG["window_size"],
    step_size=CONFIG["step_size"],
    train_split=CONFIG["train_split"],
    batch_size_train=CONFIG["batch_size_train"],
    batch_size_test=CONFIG["batch_size_test"],
)

print(f"\nData split:")
print(f"  Training windows: {len(train_dataset)}")
print(f"  Test windows: {len(test_dataset)}")
print(f"  Window size: {CONFIG['window_size']}")
print(f"  Step size: {CONFIG['step_size']}")

## 4. Train All Models with Comparable Settings
Train CNNLSTM and TCN architectures under matched hyperparameters.

In [ ]:
# Initialize model comparison tracker
comparison = ModelComparison(output_dir="model_comparison_results")

# Define models to compare
models_to_train = [
    {
        "name": "CNNLSTM",
        "config": {
            "hidden_dim": 64,
            "lstm_layers": 2,
        },
        "is_sklearn": False,
    },
    {
        "name": "TCN",
        "config": {
            "num_layers": 4,
            "num_channels": 64,
        },
        "is_sklearn": False,
    },
    {
        "name": "RandomForest",
        "config": {
            "n_estimators": 100,
            "max_depth": 20,
            "min_samples_split": 5,
        },
        "is_sklearn": True,
    },
]

# Train each model
for model_config in models_to_train:
    result = comparison.train_model_variant(
        architecture=model_config["name"],
        train_loader=train_loader,
        test_loader=test_loader,
        train_dataset=train_dataset,
        num_features=len(IMU_FEATURES),
        num_classes=len(activity_to_id),
        num_epochs=CONFIG["num_epochs"],
        learning_rate=CONFIG["learning_rate"],
        clip_grad_norm=CONFIG["clip_grad_norm"],
        patience=CONFIG["patience"],
        device=device,
        model_kwargs=model_config["config"],
        is_sklearn=model_config["is_sklearn"],
    )

## 5. Compare Metrics and Model Performance
Aggregate results and rank models by performance.

In [ ]:
# Print comparison summary
print(comparison.compare_architectures())

# Save results to JSON
comparison.save_results()

## 6. Visualize Model Comparison
Create plots comparing model metrics, training history, and confusion matrices.

In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Metrics Comparison
metrics_names = ["accuracy", "precision", "recall", "f1_score"]
architectures = list(comparison.results.keys())

metrics_data = {metric: [comparison.results[arch]["metrics"][metric] for arch in architectures] 
                for metric in metrics_names}

x = np.arange(len(architectures))
width = 0.2

for i, metric in enumerate(metrics_names):
    axes[0, 0].bar(x + i * width, metrics_data[metric], width, label=metric)

axes[0, 0].set_xlabel("Architecture")
axes[0, 0].set_ylabel("Score")
axes[0, 0].set_title("Performance Metrics Comparison")
axes[0, 0].set_xticks(x + width * 1.5)
axes[0, 0].set_xticklabels(architectures)
axes[0, 0].legend()
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis="y", alpha=0.3)

# 2. Training Loss History
for arch in architectures:
    history = comparison.results[arch]["history"]
    axes[0, 1].plot(history["train_losses"], label=f"{arch} Train", marker="o", alpha=0.7)
    axes[0, 1].plot(history["val_losses"], label=f"{arch} Val", marker="s", alpha=0.7)

axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].set_title("Training and Validation Loss")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Training Accuracy History
for arch in architectures:
    history = comparison.results[arch]["history"]
    axes[1, 0].plot(history["train_accuracies"], label=f"{arch} Train", marker="o", alpha=0.7)
    axes[1, 0].plot(history["val_accuracies"], label=f"{arch} Val", marker="s", alpha=0.7)

axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy (%)")
axes[1, 0].set_title("Training and Validation Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Model Parameters Count
param_counts = [comparison.results[arch]["param_count"] for arch in architectures]
axes[1, 1].bar(architectures, param_counts, color=["#1f77b4", "#ff7f0e"])
axes[1, 1].set_ylabel("Number of Parameters")
axes[1, 1].set_title("Model Complexity (Parameter Count)")
axes[1, 1].grid(axis="y", alpha=0.3)

# Add parameter count labels on bars
for i, (arch, count) in enumerate(zip(architectures, param_counts)):
    axes[1, 1].text(i, count, f"{count:,}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("model_comparison_results/comparison_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

print("Comparison plot saved to model_comparison_results/comparison_metrics.png")

## 7. Visualize Confusion Matrices
Plot confusion matrices for each model to analyze class-specific performance.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Plot confusion matrices
fig, axes = plt.subplots(1, len(architectures), figsize=(12, 5))
if len(architectures) == 1:
    axes = [axes]

for idx, arch in enumerate(architectures):
    cm = np.array(comparison.results[arch]["confusion_matrix"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(id_to_activity.values()))
    disp.plot(ax=axes[idx], cmap="Blues", xticks_rotation=45)
    axes[idx].set_title(f"{arch}\n(Accuracy: {comparison.results[arch]['metrics']['accuracy']:.2%})")

plt.tight_layout()
plt.savefig("model_comparison_results/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

print("Confusion matrices saved to model_comparison_results/confusion_matrices.png")

## 8. Detailed Classification Reports
Generate per-class classification metrics for best-performing models.

In [ ]:
from train_eval import build_classification_report

# Print detailed classification reports
for arch in architectures:
    result = comparison.results[arch]
    y_true = result["y_true"]
    y_pred = result["y_pred"]
    
    print(f"\n{'='*70}")
    print(f"Classification Report: {arch}")
    print(f"{'='*70}")
    print(build_classification_report(y_true, y_pred, id_to_activity))

## 9. Save Best Model and Results
Persist the best-performing model checkpoint and comparison data.

In [ ]:
# Get best model by F1 score
best_arch, best_model = comparison.get_best_model()
print(f"\n{'='*70}")
print(f"BEST MODEL: {best_arch}")
print(f"{'='*70}")
print(f"Accuracy:  {comparison.results[best_arch]['metrics']['accuracy']:.4f}")
print(f"Precision: {comparison.results[best_arch]['metrics']['precision']:.4f}")
print(f"Recall:    {comparison.results[best_arch]['metrics']['recall']:.4f}")
print(f"F1 Score:  {comparison.results[best_arch]['metrics']['f1_score']:.4f}")
print(f"Parameters: {comparison.results[best_arch]['param_count']:,}")

# Save best model with a standard name if desired
best_model_checkpoint = comparison.results[best_arch]["best_model_path"]
print(f"\nBest model checkpoint: {best_model_checkpoint}")

# Create a summary dataframe
summary_df = pd.DataFrame({
    "Model": [arch for arch in comparison.results.keys()],
    "Accuracy": [comparison.results[arch]["metrics"]["accuracy"] for arch in comparison.results.keys()],
    "Precision": [comparison.results[arch]["metrics"]["precision"] for arch in comparison.results.keys()],
    "Recall": [comparison.results[arch]["metrics"]["recall"] for arch in comparison.results.keys()],
    "F1 Score": [comparison.results[arch]["metrics"]["f1_score"] for arch in comparison.results.keys()],
    "Parameters": [comparison.results[arch]["param_count"] for arch in comparison.results.keys()],
})

summary_df = summary_df.sort_values("F1 Score", ascending=False)
summary_df.to_csv("model_comparison_results/model_comparison_summary.csv", index=False)

print(f"\nModel Comparison Summary:")
print(summary_df.to_string(index=False))

## 10. Run Inference with Best Model
Use the best model to make predictions on new IMU sequences.

In [ ]:
# Example: Run inference on test set with best model
best_model.eval()
all_predictions = []
all_confidences = []

with torch.no_grad():
    for batch_inputs, batch_labels in test_loader:
        batch_inputs = batch_inputs.to(device)
        logits = best_model(batch_inputs)
        probabilities = torch.softmax(logits, dim=1)
        predicted_ids = logits.argmax(dim=1)
        
        all_predictions.extend(predicted_ids.cpu().tolist())
        all_confidences.extend(probabilities.max(dim=1).values.cpu().tolist())

# Create inference results dataframe
inference_results = pd.DataFrame({
    "Predicted_Activity": [id_to_activity[pred_id] for pred_id in all_predictions],
    "Predicted_ID": all_predictions,
    "Confidence": all_confidences,
})

print(f"\nInference Results (first 10 predictions):")
print(inference_results.head(10))

print(f"\nAverage Confidence: {np.mean(all_confidences):.4f}")
print(f"Min Confidence: {np.min(all_confidences):.4f}")
print(f"Max Confidence: {np.max(all_confidences):.4f}")

# Visualize confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_confidences, bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Prediction Confidence")
axes[0].set_ylabel("Frequency")
axes[0].set_title(f"Confidence Distribution ({best_arch})")
axes[0].grid(axis="y", alpha=0.3)

# Confidence by activity
activity_confidences = {}
for i, pred_id in enumerate(all_predictions):
    activity = id_to_activity[pred_id]
    if activity not in activity_confidences:
        activity_confidences[activity] = []
    activity_confidences[activity].append(all_confidences[i])

activities = list(activity_confidences.keys())
avg_confidences = [np.mean(activity_confidences[act]) for act in activities]

axes[1].bar(activities, avg_confidences, color="steelblue", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Activity")
axes[1].set_ylabel("Average Confidence")
axes[1].set_title("Average Confidence by Activity")
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylim([0, 1])
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("model_comparison_results/inference_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nInference confidence plot saved to model_comparison_results/inference_confidence.png")

## 10. Run Inference with Best Model
Use the best model to make predictions on new IMU sequences.

In [ ]:
# Get best model by F1 score
best_arch, best_model = comparison.get_best_model()
print(f"\n{'='*70}")
print(f"BEST MODEL: {best_arch}")
print(f"{'='*70}")
print(f"Accuracy:  {comparison.results[best_arch]['metrics']['accuracy']:.4f}")
print(f"Precision: {comparison.results[best_arch]['metrics']['precision']:.4f}")
print(f"Recall:    {comparison.results[best_arch]['metrics']['recall']:.4f}")
print(f"F1 Score:  {comparison.results[best_arch]['metrics']['f1_score']:.4f}")
print(f"Parameters: {comparison.results[best_arch]['param_count']:,}")

# Save best model with a standard name if desired
best_model_checkpoint = comparison.results[best_arch]["best_model_path"]
print(f"\nBest model checkpoint: {best_model_checkpoint}")

# Create a summary dataframe
summary_df = pd.DataFrame({
    "Model": [arch for arch in comparison.results.keys()],
    "Accuracy": [comparison.results[arch]["metrics"]["accuracy"] for arch in comparison.results.keys()],
    "Precision": [comparison.results[arch]["metrics"]["precision"] for arch in comparison.results.keys()],
    "Recall": [comparison.results[arch]["metrics"]["recall"] for arch in comparison.results.keys()],
    "F1 Score": [comparison.results[arch]["metrics"]["f1_score"] for arch in comparison.results.keys()],
    "Parameters": [comparison.results[arch]["param_count"] for arch in comparison.results.keys()],
})

summary_df = summary_df.sort_values("F1 Score", ascending=False)
summary_df.to_csv("model_comparison_results/model_comparison_summary.csv", index=False)

print(f"\nModel Comparison Summary:")
print(summary_df.to_string(index=False))

## 9. Save Best Model and Results
Persist the best-performing model checkpoint and comparison data.

In [ ]:
from train_eval import build_classification_report

# Print detailed classification reports
for arch in architectures:
    result = comparison.results[arch]
    y_true = result["y_true"]
    y_pred = result["y_pred"]
    
    print(f"\n{'='*70}")
    print(f"Classification Report: {arch}")
    print(f"{'='*70}")
    print(build_classification_report(y_true, y_pred, id_to_activity))

## 8. Detailed Classification Reports
Generate per-class classification metrics for best-performing models.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Plot confusion matrices
fig, axes = plt.subplots(1, len(architectures), figsize=(12, 5))
if len(architectures) == 1:
    axes = [axes]

for idx, arch in enumerate(architectures):
    cm = np.array(comparison.results[arch]["confusion_matrix"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(id_to_activity.values()))
    disp.plot(ax=axes[idx], cmap="Blues", xticks_rotation=45)
    axes[idx].set_title(f"{arch}\n(Accuracy: {comparison.results[arch]['metrics']['accuracy']:.2%})")

plt.tight_layout()
plt.savefig("model_comparison_results/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

print("Confusion matrices saved to model_comparison_results/confusion_matrices.png")

## 7. Visualize Confusion Matrices
Plot confusion matrices for each model to analyze class-specific performance.

In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Metrics Comparison
metrics_names = ["accuracy", "precision", "recall", "f1_score"]
architectures = list(comparison.results.keys())

metrics_data = {metric: [comparison.results[arch]["metrics"][metric] for arch in architectures] 
                for metric in metrics_names}

x = np.arange(len(architectures))
width = 0.2

for i, metric in enumerate(metrics_names):
    axes[0, 0].bar(x + i * width, metrics_data[metric], width, label=metric)

axes[0, 0].set_xlabel("Architecture")
axes[0, 0].set_ylabel("Score")
axes[0, 0].set_title("Performance Metrics Comparison")
axes[0, 0].set_xticks(x + width * 1.5)
axes[0, 0].set_xticklabels(architectures)
axes[0, 0].legend()
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis="y", alpha=0.3)

# 2. Training Loss History
for arch in architectures:
    history = comparison.results[arch]["history"]
    axes[0, 1].plot(history["train_losses"], label=f"{arch} Train", marker="o", alpha=0.7)
    axes[0, 1].plot(history["val_losses"], label=f"{arch} Val", marker="s", alpha=0.7)

axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].set_title("Training and Validation Loss")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Training Accuracy History
for arch in architectures:
    history = comparison.results[arch]["history"]
    axes[1, 0].plot(history["train_accuracies"], label=f"{arch} Train", marker="o", alpha=0.7)
    axes[1, 0].plot(history["val_accuracies"], label=f"{arch} Val", marker="s", alpha=0.7)

axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy (%)")
axes[1, 0].set_title("Training and Validation Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Model Parameters Count
param_counts = [comparison.results[arch]["param_count"] for arch in architectures]
axes[1, 1].bar(architectures, param_counts, color=["#1f77b4", "#ff7f0e"])
axes[1, 1].set_ylabel("Number of Parameters")
axes[1, 1].set_title("Model Complexity (Parameter Count)")
axes[1, 1].grid(axis="y", alpha=0.3)

# Add parameter count labels on bars
for i, (arch, count) in enumerate(zip(architectures, param_counts)):
    axes[1, 1].text(i, count, f"{count:,}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("model_comparison_results/comparison_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

print("Comparison plot saved to model_comparison_results/comparison_metrics.png")

## 6. Visualize Model Comparison
Create plots comparing model metrics, training history, and confusion matrices.

In [ ]:
# Print comparison summary
print(comparison.compare_architectures())

# Save results to JSON
comparison.save_results()

## 5. Compare Metrics and Model Performance
Aggregate results and rank models by performance.

In [ ]:
# Initialize model comparison tracker
comparison = ModelComparison(output_dir="model_comparison_results")

# Define models to compare
models_to_train = [
    {
        "name": "CNNLSTM",
        "config": {
            "hidden_dim": 64,
            "lstm_layers": 2,
        }
    },
    {
        "name": "TCN",
        "config": {
            "num_layers": 4,
            "num_channels": 64,
        }
    },
]

# Train each model
for model_config in models_to_train:
    result = comparison.train_model_variant(
        architecture=model_config["name"],
        train_loader=train_loader,
        test_loader=test_loader,
        train_dataset=train_dataset,
        num_features=len(IMU_FEATURES),
        num_classes=len(activity_to_id),
        num_epochs=CONFIG["num_epochs"],
        learning_rate=CONFIG["learning_rate"],
        clip_grad_norm=CONFIG["clip_grad_norm"],
        patience=CONFIG["patience"],
        device=device,
        model_kwargs=model_config["config"],
    )

## 4. Train All Models with Comparable Settings
Train CNNLSTM and TCN architectures under matched hyperparameters.

In [ ]:
# Build train/test loaders with windowed data
train_loader, test_loader, train_dataset, test_dataset = make_train_test_loaders(
    data=data,
    imu_features=IMU_FEATURES,
    window_size=CONFIG["window_size"],
    step_size=CONFIG["step_size"],
    train_split=CONFIG["train_split"],
    batch_size_train=CONFIG["batch_size_train"],
    batch_size_test=CONFIG["batch_size_test"],
)

print(f"\nData split:")
print(f"  Training windows: {len(train_dataset)}")
print(f"  Test windows: {len(test_dataset)}")
print(f"  Window size: {CONFIG['window_size']}")
print(f"  Step size: {CONFIG['step_size']}")

## 3. Preprocess Data and Build Time Windows
Create sliding windows from the time-series IMU data for supervised learning.

In [ ]:
# Load and inspect data
data = load_filtered_recordings(data_path=CONFIG["data_path"], min_recordings_per_activity=5)
activity_to_id = encode_activities(data)
clean_imu_columns(data, IMU_FEATURES)

# Dataset statistics
segment_lengths = [len(df) for df in data]
print(f"Dataset loaded:")
print(f"  Total recordings: {len(data)}")
print(f"  Unique activities: {len(activity_to_id)}")
print(f"  Activities: {list(activity_to_id.keys())}")
print(f"  Segment length - min: {min(segment_lengths)}, median: {int(np.median(segment_lengths))}, max: {max(segment_lengths)}")
print(f"  IMU features ({len(IMU_FEATURES)}): {IMU_FEATURES}")

id_to_activity = {idx: activity for activity, idx in activity_to_id.items()}

## 2. Load and Inspect Dataset
Load IMU recordings and check data structure and class distribution.

In [ ]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_pipeline import (
    IMU_FEATURES,
    clean_imu_columns,
    encode_activities,
    load_filtered_recordings,
    make_train_test_loaders,
)
from model_architecture import CNNLSTM, TCN
from model_comparison import ModelComparison
from train_eval import get_device

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configuration
CONFIG = {
    "data_path": "data",
    "window_size": 300,
    "step_size": 100,
    "train_split": 0.8,
    "batch_size_train": 32,
    "batch_size_test": 1,
    "num_epochs": 50,
    "learning_rate": 2e-4,
    "clip_grad_norm": 1.0,
    "patience": 7,
}

# Get device
device = get_device()
print(f"Using device: {device}")

## 1. Environment Setup and Reproducibility
Load dependencies, set random seeds, and configure device.

# Model Comparison: CNNLSTM vs TCN
Compare different neural network architectures for exercise classification from IMU data.